# Parte 5 — Minería de Textos: topic modeling y evolución temporal (módulo 6)

**Proyecto:** Observatorio del discurso presidencial en español
**Unidades del temario cubiertas:** 5.1 (introducción y aplicaciones), 5.2 (técnicas básicas:
frecuencias, co-ocurrencia, PMI), 5.3 (algoritmos y modelos: LSA, NMF, LDA), 5.4-5.5 (herramientas y
software) y 5.6 (más allá de la MT: dinámica temporal de tópicos).

**Restricciones respetadas:** cero IA generativa, cero deep learning, cero transformers y cero
embeddings neuronales. Todo el módulo es determinista y auditable: TF-IDF, SVD truncada, NMF, LDA
(probabilístico clásico), PMI/NPMI, coseno, asignación húngara y reglas léxicas.

Este notebook **lee los artefactos** que genera `mt_scripts/build_topics.py` y muestra el pipeline
completo; no reindexa nada: reutiliza el vocabulario, el IDF y los `doc_id` del índice de RI.

| Pieza | Dónde está |
|---|---|
| Artefactos y tablas | `mt_scripts/artifacts/` |
| Documento del módulo | `mt_scripts/mt_explained.md` |
| Pipeline | `python mt_scripts/build_topics.py` |

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ri_scripts.index import load_index
from ri_scripts.corpus import read_doc

import mt_scripts.config as config
from mt_scripts import coherence as CO
from mt_scripts import evaluate as EV
from mt_scripts import labels as LAB
from mt_scripts import matrices as M
from mt_scripts import models as MO
from mt_scripts import timeline as TL

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

ix = load_index()
surfaces = M.build_stem_surface(ix, verbose=False)
print(f"corpus: {ix.n_docs} discursos, vocabulario RI: {ix.vocab_size} términos")
for speaker, (a, b) in ix.speaker_blocks.items():
    print(f"  {speaker:32s} {b - a:5d} discursos  (doc_id {a}-{b - 1})")

## 1. Base de RI reutilizada y las tres matrices de trabajo

El módulo 1 dejó `corpus.jsonl`, `postings.npz`, `idf.npy`, `vocabulary.json` y `w_global.npz`.
`w_global` tiene las **filas normalizadas L2** (para que coseno = producto punto), lo que borra la
longitud del documento y aplana los discursos largos. Por eso MT no la usa como entrada de los
modelos, sino que reconstruye tres matrices desde `postings + idf`:

| matriz | definición | uso |
|---|---|---|
| `X` | conteos `tf_ij` | LDA, PMI, UMass |
| `W` | `tf_ij * idf_j` sin normalizar | NMF, keyness |
| `W_l2` | `W` con filas L2 = 1 | coseno, SVD, alineación |

La reconstrucción es exacta: normalizar `W` devuelve `w_global` bit a bit. Eso se comprueba aquí, no
se asume.

In [ ]:
verif = M.verify_reconstruction(ix)
print(verif)

space = M.build_space(ix)                      # vocabulario de MT, con lista de puesta en escena
space_full = M.build_space(ix, drop_boilerplate=False)   # sin filtrar (comparación)

print(f"\nMT:      {space.n_docs} docs x {space.n_terms} términos ({space.X.nnz:,} no-cero)")
print(f"sin filtro: {space_full.n_terms} términos")

removed = sorted(set(space_full.terms) & config.BOILERPLATE_TERMS)
print(f"\npuesta en escena: {len(config.BOILERPLATE_TERMS)} stems en la lista, "
      f"{len(removed)} presentes en el vocabulario y eliminados")
print("  ", ", ".join(removed))

W = space.W
norm_w = np.sqrt(np.asarray(W.multiply(W).sum(axis=1)).ravel())
norm_l2 = np.sqrt(np.asarray(space.W_l2.multiply(space.W_l2).sum(axis=1)).ravel())
print("\nfilas de W (tf*idf) sin normalizar:", np.round(norm_w[:6], 1))
print("filas de W_l2:", np.round(norm_l2[:6], 4))
print(f"df de los términos conservados: min={space.df.min()}, max={space.df.max()}")

**Por qué se elimina la puesta en escena.** El corpus confunde país = persona (restricción 1 del plan
base). Si dejamos "chilenos", "Chile", "compatriotas", "presidente", "interlocutora", el modelo agrupa
discursos **por quién habla** y no por lo que dice: el tópico dominante de Piñera sale "chilenos ·
Chile · compatriotas". La lista `config.BOILERPLATE_TERMS` (cargos, gentilicios de autorreferencia,
nombres de los cinco hablantes, dícticos) se elimina del vocabulario de MT, y se documenta como decisión.

## 2. Los tres modelos (temario 5.3)

- **LSA** — SVD truncada de `W_l2`: `W_l2 ≈ U Σ Vᵀ`. Las `k` primeras componentes son los tópicos.
  Las cargas tienen signo arbitrario, así que se toma `|V|` y se normaliza. Aporta la línea base no
  probabilística y la varianza explicada.
- **NMF** — `W_l2 ≈ H V` con `H, V ≥ 0`, minimizando `‖W_l2 − H V‖²_F`. Los tópicos son aditivos y
  dispersos, que es lo que hace legibles las listas de términos.
- **LDA** — modelo generativo: cada documento es una mezcla `θ_d ~ Dir(α)` y cada término se genera
  desde `z ~ Cat(θ_d)`, `w ~ Cat(φ_z)` con `φ_z ~ Dir(β)`. Se ajusta por inferencia variacional.

Los tres se ajustan sobre el **mismo** `Space` y devuelven la misma interfaz: `topic_word` (k × V),
`doc_topic` (N × k) y vectores de tópico en el vocabulario global para poder alinearlos.

In [ ]:
demo = {}
for family in ("lsa", "nmf", "lda"):
    t0 = time.time()
    demo[family] = MO.fit(family, space, 8)
    m = demo[family]
    secs = time.time() - t0
    extra = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.extras.items()}
    print(f"{MO.FAMILY_LABEL[family]:20s} k=8  {secs:6.1f}s  {extra}")
    print("   T0:", ", ".join(LAB.readable(t, surfaces) for t in m.top_terms(0, 8)))
    print("   T1:", ", ".join(LAB.readable(t, surfaces) for t in m.top_terms(1, 8)))

## 3. Coherencia desde cero: PMI, NPMI, c_v y UMass (temario 5.2)

El espacio de muestreo son los pares de tokens a distancia `d < W` (`W = 10`). Con `m_ij` pares
ordenados y `T = Σ m_ij`:

$$p_{ij} = \frac{m_{ij}}{T},\qquad p_i = \frac{\sum_j m_{ij}}{T},\qquad
\mathrm{PMI}(i,j) = \log\frac{p_{ij}}{p_i\,p_j},\qquad
\mathrm{NPMI}(i,j) = \frac{\mathrm{PMI}(i,j)}{-\log p_{ij}} \in [-1,1].$$

`c_v` mide la **confirmación semántica** entre las palabras de un tópico: cada término se representa
por su vector de NPMI restringido a las palabras del tópico y se promedian los cosenos entre pares.

`UMass` usa co-ocurrencia por documento, con los términos ordenados por df decreciente:

$$\mathrm{UMass} = \frac{2}{n(n-1)}\sum_{i<j}\log\frac{D(i,j)+1}{D(i)}.$$

Se calcula para los top-10 términos de cada tópico. Comparar las tres familias con la misma métrica es
lo que justifica la elección, en vez de "usamos LDA porque sí".

In [ ]:
cand = np.unique(np.concatenate([m.top_ids(t, 10) for m in demo.values() for t in range(m.k)]))
cooc = CO.build_cooc(ix, cand)

filas = []
for family, m in demo.items():
    met = CO.evaluate_model(m, cooc, cand)
    filas.append({"familia": MO.FAMILY_LABEL[family], **{k: met[k] for k in
                  ("c_v", "umass", "diversity", "exclusivity", "mean_top_weight")}})
display(pd.DataFrame(filas).round(3))

print("Pares con mayor NPMI en el corpus (ventana 10):")
display(pd.DataFrame(CO.strongest_pairs(cooc, ix, 12)).round(3))

## 4. Elección de `k`: barrido y regla de un error estándar

La coherencia sube de forma casi monótona con `k`: más tópicos = listas de términos más específicas.
Quedarse con el máximo favorece siempre el `k` mayor y fragmenta el modelo. La regla usada es:

1. familia = la de mayor `c_v` (con diversidad ≥ 0,60 como filtro);
2. dentro de la familia, `SE = σ(c_v por tópico)/√k` del mejor modelo;
3. se elige el **menor** `k` con `c_v ≥ c_v_max − max(SE, 0,05)`.

La banda de 0,05 es del orden del error estándar observado (≈ 0,048): diferencias menores no son
distinguibles con 4.531 documentos. Es una regla de parsimonia, no una excusa: abajo se muestra qué
pasa con cada `k`.

In [ ]:
sweep = json.load(open(config.SWEEP_JSON, encoding="utf-8"))
rows = pd.DataFrame([{k: r[k] for k in ("family", "k", "c_v", "umass", "diversity", "exclusivity", "seconds")}
                     for r in sweep["rows"]])
chosen = sweep["meta"]["chosen"]
print("Modelo elegido:", chosen["family"], "k =", chosen["k"], "|", chosen["criterion"])
print("c_v máxima:", round(chosen["best_in_family"]["c_v"], 3),
      "en k =", chosen["best_in_family"]["k"],
      "| SE =", round(chosen["best_in_family"]["se"], 3),
      "| umbral =", round(chosen["threshold"], 3))

display(rows.pivot(index="k", columns="family", values="c_v").round(3))
display(rows.sort_values(["family", "k"]).round(3).reset_index(drop=True))

## 5. Tópicos del modelo elegido (temario 5.3)

Cada tópico se presenta con sus términos de mayor peso (forma superficial más frecuente del stem), la
etiqueta automática y, si el gazetteer de `labels.py` encuentra ≥ 2 anclas en el top-20 (con al menos
una en el top-10), una **etiqueta manual auditable** con el número de anclas que la dispararon. Nunca
se inventa un nombre sin anclas que lo respalden.

In [ ]:
tg = json.load(open(config.TOPICS_JSON, encoding="utf-8"))
topics = tg["topics"]
label_rows = pd.DataFrame(tg["label_rows"])

shares = pd.DataFrame(tg["speaker_shares"])
shares["hablante"] = shares["speaker"].str.split().str[0]
piv = shares.pivot_table(index="topic", columns="hablante", values="share").fillna(0)

tabla = pd.DataFrame([{
    "topic": t["topic"],
    "nombre": t["name"],
    "términos": ", ".join(x["surface"] for x in t["top_terms"][:8]),
    "anclas": len(t["manual_anchors"]),
    "participación global": round(t["share"], 3),
} for t in topics]).set_index("topic")
display(tabla.join(piv.round(3)))

In [ ]:
# Evidencia: los documentos con mayor θ en un tópico, con un extracto del discurso
topico = 6   # economía, inflación y fiscal en el modelo elegido
info = topics[topico]
print(f"T{topico} — {info['name']}")
print("términos:", ", ".join(f"{x['surface']}({x['weight']:.3f})" for x in info["top_terms"][:8]))
for doc in info["top_docs"][:3]:
    print(f"\n  {doc['speaker']} · {doc['date']} · θ={doc['theta']:.3f} · doc_id={doc['doc_id']}")
    print("  ", doc["snippet"][:280])

In [ ]:
# θ de un discurso concreto: ¿de qué habla y cuánto de cada tópico?
doc_topics = pd.DataFrame([json.loads(l) for l in open(config.DOC_TOPICS_JSONL, encoding="utf-8")])
fila = doc_topics[doc_topics.doc_id == 2295].iloc[0]   # asunción de Milei
print(f"{fila.speaker} · {fila.date} · {ix.titles[2295][:90]}")
dist = pd.DataFrame({"topic": range(chosen["k"]), "theta": fila.theta,
                     "nombre": [t["name"] for t in topics]}).sort_values("theta", ascending=False)
display(dist.head(5).round(4).reset_index(drop=True))

## 6. Evolución temporal, vía A: un modelo, θ agregada por periodo

Un único modelo para todo el corpus; cada documento recibe su `θ`. Se promedia `θ` por hablante-año
(y hablante-trimestre). Como cada fila de `θ` suma 1, la media es una **participación** y la
comparación entre hablantes es proporcional: el desbalance 2.294 : 30 no domina la tabla.

Límite de la vía A: los tópicos son los mismos en todos los periodos, así que un tema que nace en 2024
o cambia de vocabulario no puede manifestarse. Para eso está la vía B (§8).

In [ ]:
tl = json.load(open(config.TIMELINE_JSON, encoding="utf-8"))
serie = pd.DataFrame(tl["year"])
serie["casilla"] = serie["speaker"].str.split().str[0] + " " + serie["period"]
anual = serie.pivot_table(index="topic", columns="casilla", values="share").fillna(0)

fig, ax = plt.subplots(figsize=(15, 5.5))
sns.heatmap(anual, cmap="viridis", ax=ax, cbar_kws={"label": "participación media de θ"})
ax.set_title("Participación de cada tópico por hablante-año")
ax.set_ylabel("tópico")
plt.tight_layout()
plt.show()

In [ ]:
# La misma serie en formato largo: trayectoria trimestral de dos tópicos en un hablante
trim = pd.DataFrame(tl["quarter"])
for topic, speaker in [(6, "Javier Milei"), (5, "Andrés Manuel López Obrador")]:
    sub = trim[(trim.topic == topic) & (trim.speaker == speaker)].sort_values("period")
    if sub.empty:
        continue
    plt.figure(figsize=(10, 2.6))
    plt.plot(sub.period, sub.share, marker="o", ms=3)
    plt.title(f"T{topic} — {topics[topic]['name']} · {speaker}")
    plt.xticks(rotation=90, fontsize=7)
    plt.ylabel("participación")
    plt.tight_layout()
    plt.show()

## 7. Ventana limpia 2023-2024 (restricción 3 del plan base)

Los periodos de los cinco líderes son casi disjuntos. La única ventana con 4 de los 5 activos es
2023-2024 (Maduro solo tiene un discurso en 2023). Cualquier comparación **entre** líderes se hace
aquí; las series largas se leen dentro de cada hablante.

In [ ]:
limpia = pd.DataFrame(tl["year_clean"])
limpia["casilla"] = limpia["speaker"].str.split().str[0] + " " + limpia["period"]
piv = limpia.pivot_table(index="topic", columns="casilla", values="share").fillna(0)
piv.index = [f"T{t} {topics[t]['name'][:34]}" for t in piv.index]
display(piv.round(3))

Lectura rápida: AMLO se reparte entre protección social, seguridad, infraestructura y su tópico de
prensa matutina; Sánchez entre política exterior/instituciones y economía; Milei concentra ~0,7-0,8 de
su θ en el tópico económico (inflación, PIB, ajuste). Es la evidencia de que el modelo capta estilos
temáticos distintos y no sólo un "promedio" del corpus.

## 8. Vía B: un modelo por hablante-año y alineación de tópicos

Se ajusta un modelo NMF por `(hablante, año)` con ≥ 30 discursos. Los tópicos de años consecutivos se
emparejan por el coseno de sus **conjuntos binarios** de los 50 términos de mayor peso
(`sim = |A∩B| / sqrt(|A|·|B|)`), con **asignación óptima húngara** (`linear_sum_assignment` sobre
`-similitud`). La **deriva** es esa similitud del mismo tópico en dos años; `sim ≥ 0,25` (≈ 12 de 50
términos compartidos) cuenta como tópico emparejado. Lo demás son tópicos nuevos o perdidos.

¿Por qué binarios? Con los pesos continuos, dos tópicos del mismo tema comparten pocos términos en la
cabeza y el coseno cae a 0,01-0,05: indistinguible de dos tópicos ajenos (0,00-0,01). Con conjuntos,
un mismo tema comparte 20-29 de 50 términos (0,40-0,58) y uno ajeno < 10 (< 0,20). El emparejamiento
húngaro evita además que dos tópicos caigan en el mismo.

Aquí sí se pueden ver tópicos que nacen (COVID en 2020), se transforman o desaparecen.

In [ ]:
win = json.load(open(config.WINDOW_TOPICS_JSON, encoding="utf-8"))
resumen = pd.DataFrame(win["summary"])
display(resumen.round(3).reset_index(drop=True))

deriva = pd.DataFrame(win["drift"])
display(deriva[deriva.matched].sort_values("similarity", ascending=False).head(12).round(3).reset_index(drop=True))

In [ ]:
# Cadenas: seguir un tópico a través de los años de un mismo hablante
por_ventana = {(it["speaker"], it["year"]): it for it in win["items"]}

def _final_terms(c):
    item = por_ventana.get((c["speaker"], str(c.get("year_end", c["year_start"]))))
    if item is None:
        return ""
    topic = c.get("topic_end", c["topic_start"])
    return ", ".join(item["topics"][topic]["top_terms"][:5])

cadenas = pd.DataFrame([{
    "hablante": c["speaker"],
    "año inicio": c["year_start"],
    "año fin": c.get("year_end", c["year_start"]),
    "años": c["n_years"],
    "similitud media": round(c["mean_similarity"], 3),
    "tópico al final": _final_terms(c),
} for c in win["chains"]])
display(cadenas.sort_values("similitud media", ascending=False).head(12).reset_index(drop=True))

# Ejemplo concreto: evolución de un tópico de AMLO en sus ventanas anuales
amlo = [it for it in win["items"] if it["speaker"].startswith("Andrés")]
for it in amlo:
    print(f"\n{it['year']} ({it['docs']} discursos)")
    for t in it["topics"][:3]:
        print(f"   T{t['topic']} ({t['share']:.2f}): {', '.join(t['top_terms'][:7])}")

## 9. Tópicos por hablante (modelo independiente)

Un modelo por hablante, `k = 8`. Es la lectura correcta del dataset: 5 hablantes, 1 por país, país =
persona. Maduro tiene 30 discursos y se marca como **baja confianza**; no se mezcla en agregados sin
nota.

In [ ]:
spk = json.load(open(config.SPEAKER_TOPICS_JSON, encoding="utf-8"))["items"]
filas = []
for item in spk:
    for t in item["topics"][:4]:
        filas.append({
            "hablante": item["speaker"].split()[0],
            "discursos": item["docs"],
            "confianza": "baja" if item["low_confidence"] else "normal",
            "tópico": t["topic"],
            "nombre": t["name"],
            "participación": round(t["share"], 3),
            "términos": ", ".join(t["top_terms"][:7]),
        })
display(pd.DataFrame(filas).reset_index(drop=True))

## 10. Evaluación (sin gold de tópicos: cuatro señales)

No hay un gold de tópicos para 4.531 discursos, así que se combinan cuatro señales independientes:

1. **intrínseca** — `c_v`, UMass, diversidad y exclusividad (§3 y §4);
2. **estabilidad** — remuestreo del 80 % de los documentos, reajuste y Jaccard del top-10 tras
   alinear tópicos por coseno;
3. **anclas** — el gold anual del módulo 5 (RAT) lista 7-8 temas clave de 4 casos; se mide cuántos
   cubren los 3 tópicos dominantes del año;
4. **gold manual** — 47 discursos etiquetados a mano con su área temática
   (`gold/topics_gold.json`); se mide **pureza** (todos los discursos de un área caen en el mismo
   tópico dominante), invariante a la numeración de tópicos.

In [ ]:
ev = json.load(open(config.EVAL_JSON, encoding="utf-8"))

print("Anclas del gold anual del módulo 5")
display(pd.DataFrame([{
    "caso": f"{a['speaker']} {a['year']}",
    "discursos": a["docs"],
    "recall": round(a["topic_recall"], 3),
    "anclas cubiertas": sum(a["hits"].values()),
    "anclas": len(a["hits"]),
    "no cubiertas": ", ".join(k for k, v in a["hits"].items() if not v),
} for a in ev["anchors"]]).reset_index(drop=True))

print(f"\nPureza gold manual: {ev['purity']['purity']:.3f} "
      f"(pureza@2 = {ev['purity']['purity_at2']:.3f}, n = {ev['purity']['n']})")
nombres = {t["topic"]: t["name"] for t in topics}
display(pd.DataFrame([{
    "área temática (manual)": r["theme"],
    "discursos": r["docs"],
    "tópico modal": f"T{r['modal_topic']} {nombres.get(r['modal_topic'], '')[:38]}",
    "coincidencia": round(r["modal_share"], 3),
} for r in ev["purity"]["by_theme"]]).reset_index(drop=True))

In [ ]:
print("Sensibilidad a k (misma familia): coherencia vs pureza vs anclas")
display(pd.DataFrame(ev["sensitivity_k"]).round(3).reset_index(drop=True))

stab = ev["stability"]
if stab:
    print(f"\nEstabilidad: {stab['runs']} remuestreos al {stab['subsample']:.0%} → "
          f"Jaccard = {stab['jaccard_mean']:.3f} ± {stab['jaccard_std']:.3f}")
    display(pd.DataFrame({
        "tópico": list(stab["jaccard_per_topic"].keys()),
        "nombre": [nombres[int(t)] for t in stab["jaccard_per_topic"]],
        "Jaccard": [round(v, 3) for v in stab["jaccard_per_topic"].values()],
    }).reset_index(drop=True))

La lectura honesta de la tabla de sensibilidad: `k = 15` tiene la `c_v` más alta (0,492) pero **peor
pureza** (0,468) y más tópicos casi duplicados. `k = 6` tiene la pureza más alta (0,638) pero cubre
menos anclas (0,344): es demasiado grueso. `k = 10` queda en el punto de equilibrio (0,553 de pureza,
0,469 de anclas, estabilidad 0,86). La regla 1-SE elige precisamente ese punto.

Ojo con la interpretación de la pureza: las áreas del gold son de "tema principal", y muchos discursos
son multimodales. La pureza@2 (0,66) refleja eso. Y varias áreas (economía, infraestructura) están
repartidas entre tópicos de distintos hablantes porque el tema se tiñe del estilo de cada uno: es el
confundido país = persona del dataset, no un fallo del modelo.

## 11. Resultados sobre el corpus (resumen)

- **Barrido.** NMF domina a LSA y LDA en `c_v` con la misma preparación: 0,44-0,49 frente a 0,13-0,14
  y 0,14-0,28. LSA produce componentes poco interpretables (términos con signo) y LDA necesita más
  iteraciones para acercarse; con 12 iteraciones en línea no alcanza.
- **Modelo elegido.** NMF, `k = 10`, `c_v = 0,443`, diversidad 0,94, exclusividad 0,63.
- **Tópicos.** Diez tópicos con lectura clara: seguridad y justicia (AMLO), política exterior
  (Sánchez), pandemia/vacunas, protección social, seguridad chilena (Piñera), infraestructura
  (Tren Maya), economía/inflación (Milei), salud/IMSS, Cataluña/instituciones y precios de la
  gasolina (AMLO).
- **Temporal.** La vía A muestra el COVID entrando en 2020 (0,05-0,23) y saliendo después; la vía B
  muestra cómo el tópico de vacunas se separa del de pandemia en 2021 y cómo AMLO gira hacia
  infraestructura/energía en 2022-2024.
- **Evaluación.** `c_v = 0,443`, UMass = -1,058, Jaccard de estabilidad 0,858 ± 0,175, anclas 0,469,
  pureza 0,553 (pureza@2 0,660).

Las tablas completas están en `mt_scripts/artifacts/tables.md` y el detalle matemático en
`mt_scripts/mt_explained.md`.

## 12. Limitaciones

1. **País = persona**: cada tópico mezcla tema y estilo del hablante. Se mitiga eliminando la puesta en
   escena y reportando siempre por hablante, pero no se elimina: los tópicos de Milei son "Milei
   hablando de economía".
2. **Desbalance**: Maduro tiene 30 discursos; su modelo (k = 6) y sus participaciones se reportan con
   nota de baja confianza y nunca en un agregado sin advertencia.
3. **Periodos disjuntos**: la comparación entre líderes se limita a 2023-2024.
4. **NMF no es probabilístico**: `θ` es una descomposición, no una posterior. La lectura de
   "participación" es descriptiva. LDA sí es probabilístico y se reporta como contraste.
5. **La coherencia c_v depende de la ventana** (10 tokens) y del vocabulario podado; se documenta y se
   reporta UMass, que usa otra unidad (documento), como segunda opinión.
6. **El gold manual es pequeño** (47 discursos, 12 áreas) y fue etiquetado con título + inicio del
   discurso. La pureza es una señal, no una verdad.
7. **Duplicados**: hay discursos repetidos en el corpus (módulo 9 del plan, pendiente). Inflan ligeramente
   los tópicos; no se deduplicó aquí para no alterar los `doc_id` de RI.
8. **El resumen de la vía B es más frágil** en ventanas de 50-90 discursos (Piñera 2022, Milei 2024-25):
   el número de tópicos se recorta y la alineación es más ruidosa.

## 13. Reproducir

```bash
source ../env/bin/activate
python mt_scripts/build_topics.py          # ~3-4 min, escribe todos los artefactos
python mt_scripts/build_topics.py --force  # reconstruye tokens y mapa stem->superficie
python mt_scripts/build_topics.py --families nmf --k 6,8 --skip-stability   # prueba rápida
```

In [ ]:
for path in sorted(config.ARTIFACTS.iterdir()):
    if path.is_file():
        print(f"{path.name:26s} {path.stat().st_size / 1e6:8.2f} MB")